# Klasyfikator w dwie minuty

In [1]:
FINAL_EVALUATION_MODE = False

Poniżej zapewniono kod do zapisywania oraz ewaluacji modelu, a także ładowania danych. Możesz, ale nie musisz z niego korzystać.

Poniższe klasy oraz metody można dowolnie modyfikować, należy jedynie pamiętać aby metoda `finetune_and_predict()` przyjmowała cztery argumenty `(X_train_small, y_train_small, X_test, model_path)`

## Kod startowy

Na początek zaimportujmy wszystkie niezbędne biblioteki, zdefiniujmy instancje klas obsługujące dane (`DataLoader`), niezbędne hiperparametry oraz funkcję do zapisywania modeli.

In [2]:
import os
import random
import warnings

import gdown
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")

In [13]:
# NIE ZMIENIAJ TYCH WARTOŚCI
DEVICE = torch.device("cuda")
MODEL_FP = "./encoder.pt"
DATA_PATH = 'data/'

In [96]:
# THESE VALUES CAN BE CHANGED
SEED =  42
EPOCHS = 200
N_CLASS = 6
N_CHANNEL = 3
N_LENGTH = 206
PROJECTION_DIM = 64
LOGISTIC_BATCH_SIZE = 64
LOGISTIC_EPOCHS = 50
FINETUNE_SEED = 42

In [6]:
if not FINAL_EVALUATION_MODE:
    ! gdown https://drive.google.com/uc?id=1RxxsAQ6memLe4B0fePJD4cV_SIuRmhmw
    # ! gdown https://drive.google.com/uc?id=1K7873pSrnIEvQe4HxhvD0S6Ur-7bEmfm
    # ! gdown https://drive.google.com/uc?id=16ynUNF1WicH37L5iJszsfvVc-NJPzz8y
    ! unzip self_supervised.zip
    

Downloading...
From: https://drive.google.com/uc?id=1RxxsAQ6memLe4B0fePJD4cV_SIuRmhmw
To: c:\Users\raian\source\repos\AI\IOAI_prep\poland\1_final\self_supervised\self_supervised.zip

  0%|          | 0.00/25.1M [00:00<?, ?B/s]
  2%|▏         | 524k/25.1M [00:00<00:06, 3.90MB/s]
  6%|▋         | 1.57M/25.1M [00:00<00:03, 6.09MB/s]
 10%|█         | 2.62M/25.1M [00:00<00:03, 6.72MB/s]
 15%|█▍        | 3.67M/25.1M [00:00<00:03, 7.01MB/s]
 19%|█▉        | 4.72M/25.1M [00:00<00:02, 7.71MB/s]
 23%|██▎       | 5.77M/25.1M [00:00<00:02, 8.23MB/s]
 27%|██▋       | 6.82M/25.1M [00:00<00:02, 8.61MB/s]
 31%|███▏      | 7.86M/25.1M [00:01<00:01, 8.87MB/s]
 36%|███▌      | 8.91M/25.1M [00:01<00:01, 9.00MB/s]
 40%|███▉      | 9.96M/25.1M [00:01<00:01, 9.00MB/s]
 46%|████▌     | 11.5M/25.1M [00:01<00:01, 9.49MB/s]
 50%|█████     | 12.6M/25.1M [00:01<00:01, 9.47MB/s]
 54%|█████▍    | 13.6M/25.1M [00:01<00:01, 9.49MB/s]
 59%|█████▊    | 14.7M/25.1M [00:01<00:01, 9.51MB/s]
 63%|██████▎   | 15.7M/25.1M [00

In [59]:
class CustomTensorDataset(Dataset):
    """TensorDataset with support of transforms."""
    def __init__(self, data, transform_A=None, transform_B=None):
        assert all(data[0].shape[0] == item.shape[0] for item in data)
        self.data = data
        self.transform_A = transform_A
        self.transform_B = transform_B

    def __getitem__(self, index):
        x = torch.tensor(self.data[0][index])

        if self.transform_A:
            x1 = self.transform_A(x)
        else:
            x1 = x
        if self.transform_B:
            x2 = self.transform_B(x)
        else:
            x2 = x
        y = self.data[1][index]
        return x1.float(), x2.float(), torch.tensor(y)

    def __len__(self):
        return self.data[0].shape[0]

In [8]:
def setup_seed(seed=42):
    """Setup seed for the reproducipility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

## Solution Framework

### Models: Encoder and Classifier

Below, an example (and not very effective) solution to the problem is implemented. A simple encoder is presented, the goal of which is to create a useful data representation that can then be used by a simple classifier such as a multilayer perceptron.

In [84]:
# Improve the class below!

class SimpleEncoder(nn.Module):
    def __init__(self, projection_dim, n_channel, n_length=240):
        super(SimpleEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channel, 32, kernel_size=7,padding=3, stride=2, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(True),
            nn.Conv1d(32, 64, kernel_size=5,padding=2, stride=2, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(True),
            nn.Conv1d(64, 128, kernel_size=3,padding=1, stride=2, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(True),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        self.projector = nn.Sequential(
            nn.Linear(128, projection_dim),
        )

    def forward(self, x_i):
        h_i = self.encoder(x_i.squeeze(1))
        z_i = self.projector(h_i)
        return h_i, z_i

    def save_model(self, model):
        torch.save(model.state_dict(), MODEL_FP)

In [10]:
# Improve the class below!

class MLPClassifier(nn.Module):
    def __init__(self, n_features, n_classes):
        super(MLPClassifier, self).__init__()
        n_dim = n_features // n_classes // 2 * n_classes

        self.model = nn.Sequential(
            nn.Linear(n_features, n_dim),
            nn.ReLU(),
            nn.Linear(n_dim, n_classes)
        )

    def forward(self, x):
        return self.model(x)

In [82]:
from torchinfo import summary
encoder_model = SimpleEncoder(
        projection_dim=PROJECTION_DIM,
        n_channel=N_CHANNEL,
        n_length=N_LENGTH
    )
summary(encoder_model, [64,1,3,206])

torch.Size([64, 3, 206])
torch.Size([64, 128])


Layer (type:depth-idx)                   Output Shape              Param #
SimpleEncoder                            [64, 128]                 --
├─Sequential: 1-1                        [64, 128]                 --
│    └─Conv1d: 2-1                       [64, 32, 103]             672
│    └─BatchNorm1d: 2-2                  [64, 32, 103]             64
│    └─ReLU: 2-3                         [64, 32, 103]             --
│    └─Conv1d: 2-4                       [64, 64, 52]              10,240
│    └─BatchNorm1d: 2-5                  [64, 64, 52]              128
│    └─ReLU: 2-6                         [64, 64, 52]              --
│    └─Conv1d: 2-7                       [64, 128, 26]             24,576
│    └─BatchNorm1d: 2-8                  [64, 128, 26]             256
│    └─ReLU: 2-9                         [64, 128, 26]             --
│    └─AdaptiveAvgPool1d: 2-10           [64, 128, 1]              --
│    └─Flatten: 2-11                     [64, 128]                 --
├─Se

### Encoder Training
If we want to train the encoder, we can use the following function. In our less-than-efficient solution, we leave it with random weights.

In [55]:
class AddGaussianNoise(nn.Module):
    def __init__(self, std):
        super().__init__()
        self.std = std

    def forward(self, x):
        return x + torch.randn_like(x) * self.std
    
class RandomScaling(nn.Module):
    def __init__(self, scale_range):
        super().__init__()
        self.scale_range = scale_range

    def forward(self, x):
        scale_factor = torch.empty(1).uniform_(*self.scale_range)
        return x * scale_factor
    
class TimeMasking(nn.Module):
    def __init__(self, mask_ratio):
        super().__init__()
        self.mask_ratio = mask_ratio

    def forward(self, x):
        mask_len = int(x.shape[-1] * self.mask_ratio)
        mask_begin = torch.randint(low=0, high=x.shape[-1] - mask_len, size=(1,)).item()
        x[:,:,mask_begin:mask_begin + mask_len] = 0
        return x
    
class RandomCrop(nn.Module):
    def __init__(self, crop_ratio):
        super().__init__()
        self.crop_ratio = crop_ratio

    def forward(self, x):
        crop_len = int(x.shape[-1] * self.crop_ratio)
        crop_begin = torch.randint(low=0, high=x.shape[-1] - crop_len, size=(1,)).item()
        cropped_x = x[:,:,crop_begin:crop_begin + crop_len]
        return F.interpolate(cropped_x, x.shape[-1], mode='linear', align_corners=False)

train_transform_A = torch.nn.Sequential(
    AddGaussianNoise(std=0.05),
    RandomScaling(scale_range=(0.9, 1.1)),
)

train_transform_B = torch.nn.Sequential(
    TimeMasking(mask_ratio=0.1),
    RandomCrop(crop_ratio=0.85),
)

In [105]:
from tqdm.notebook import tqdm
LR = 1e-3

class ContrastiveLoss(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x_i, x_j, temp=1):
        labels = torch.arange(x_i.shape[0]).to(DEVICE)
        sim = x_i @ x_j.T / temp
        return F.cross_entropy(sim, labels)

def train(unlabelled_train_loader, encoder_model):
    """Test the model (encoder) based on unlabeled data."""
    optimizer = torch.optim.AdamW(encoder_model.parameters(), lr=LR)
    criterion = ContrastiveLoss()
    loss_epoch = []
    pbar = tqdm(range(LOGISTIC_EPOCHS), desc="Training")
    for epoch in pbar:
        for step, (x_i, x_j, _) in enumerate(unlabelled_train_loader):
            optimizer.zero_grad()
            x_i, x_j = x_i.to(DEVICE), x_j.to(DEVICE)
            
            h_i, out_i = encoder_model(x_i)
            h_j, out_j = encoder_model(x_j)

            loss = criterion(out_i, out_j)
            loss.backward()
            optimizer.step()
            loss_epoch.append(loss.item())
        mean_loss = sum(loss_epoch) / len(loss_epoch)
        pbar.set_postfix(loss = f'{mean_loss:.4f}')
    
if not FINAL_EVALUATION_MODE:
    X_train_big = torch.load(DATA_PATH + "train_x_big.pt", weights_only=False)
    train_dataset = CustomTensorDataset((X_train_big, torch.zeros(X_train_big.shape)), train_transform_A, train_transform_B)

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=LOGISTIC_BATCH_SIZE,
        shuffle=True,
        drop_last=True,
    )

    encoder_model = SimpleEncoder(
        projection_dim=PROJECTION_DIM,
        n_channel=N_CHANNEL,
        n_length=N_LENGTH
    )

    encoder_model = encoder_model.to(DEVICE)
    train(train_loader, encoder_model)
    encoder_model.save_model(encoder_model)

Training:   0%|          | 0/50 [00:00<?, ?it/s]

### Trening klasyfikatora

In [106]:
# Please improve the function below!

def finetune(labelled_data_loader, encoder, classifier):
    """Train models (encoder, classifier) on labeled data."""
    optimizer = torch.optim.Adam(classifier.parameters(), lr=3e-4)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(LOGISTIC_EPOCHS):
        loss_epoch = []
        encoder.train()
        classifier.train()
        for n_batch, (x, x_aug, y) in enumerate(labelled_data_loader):
            optimizer.zero_grad()
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            with torch.no_grad():
                h, z = encoder(x)
            output = classifier(h).to(DEVICE)
            y = y.squeeze(-1).type(torch.LongTensor).to(DEVICE)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            loss_epoch.append(loss.item())
        mean_loss = sum(loss_epoch) / len(loss_epoch)

    return mean_loss

### Training and Prediction Function
Your score will be calculated based on the function below. Remember to check the formal correctness of your solution with the validation script!

In [112]:
# Popraw poniższą funkcję!

def finetune_and_predict(X_train_small, y_train_small, X_test, model_path):
    """During testing, we will run this function with test data.

    Do not change the signature of this function, i.e., the number, names, and order of arguments.

    Arguments:

    X_train_small -- the training data tensor of dimensions (N, 1, 3, 206), where N is the number of independent measurements in the set,

    y_train_small -- the label vector for the training set of length N,

    X_test -- the test data tensor of dimensions (M, 1, 3, 206), where M is the number of independent measurements in the set,

    Return:
    The function should return a vector of length M of predicted labels for the X_test set.
    """
    setup_seed(FINETUNE_SEED)

    train_dataset = CustomTensorDataset((X_train_small, y_train_small))

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=LOGISTIC_BATCH_SIZE,
        shuffle=True,
        drop_last=True,
    )

    encoder_model = SimpleEncoder(
        projection_dim=PROJECTION_DIM,
        n_channel=N_CHANNEL,
        n_length=N_LENGTH
    )

    encoder_model.load_state_dict(torch.load(model_path, map_location=DEVICE.type))
    encoder_model = encoder_model.to(DEVICE)
    n_classes = N_CLASS
    classifier = MLPClassifier(128, n_classes)
    classifier = classifier.to(DEVICE)
    mean_loss_train = finetune(train_loader, encoder_model, classifier)
    x = torch.tensor(X_test).to(DEVICE)
    h, z = encoder_model(x)
    output = classifier(h).to(DEVICE)
    predicted = output.argmax(1)
    assert len(predicted.shape) == 1
    assert len(predicted) == len(X_test)
    assert predicted.dtype == torch.int64
    return predicted

## Ewaluacja

Twój kod zostanie zewaluowany w sposób podobny do poniższego. Pamiętaj, że w testowym skrypcie, zastąpione zostaną wszystkie poniższe datasety: zarówno `X_train_small`, `y_train_small`, jak i `X_val`.

In [113]:
X_val = torch.load(DATA_PATH + "val_x.pt", weights_only=False)
y_val = torch.load(DATA_PATH + "val_y.pt", weights_only=False)
if not FINAL_EVALUATION_MODE:
    X_train_small = torch.load(DATA_PATH + "train_x_small.pt", weights_only=False)
    y_train_small = torch.load(DATA_PATH + "train_y_small.pt", weights_only=False)
    pred = finetune_and_predict(X_train_small, y_train_small, X_val, MODEL_FP)
    print("Accuracy:", accuracy_score(y_val, pred.cpu().numpy()))

Accuracy: 0.8946295037389531
